# Crossed and nested random effects

Models with more than one random-effects term: the **fully crossed** `Penicillin` design
(plates × samples), the **nested** `Pastes` design (casks within batches) with explicit and
implicit nesting, the `isNested()` check, the large **partially crossed** `InstEval` data
(students × lecturers, 73 421 ratings), and the structure of the model matrices
(`getME(m, "Zt")`, `getME(m, "Lambdat")`, `getME(m, "L")`).

## Source

- lme4 help pages [`?Penicillin`](https://github.com/lme4/lme4/blob/master/man/Penicillin.Rd),
  [`?Pastes`](https://github.com/lme4/lme4/blob/master/man/Pastes.Rd),
  [`?InstEval`](https://github.com/lme4/lme4/blob/master/man/InstEval.Rd), and
  `reformulas::isNested` (re-exported by lme4).
- Bates, Mächler, Bolker & Walker (2015), *Fitting Linear Mixed-Effects Models Using lme4*,
  J. Stat. Softw. 67(1), §2.2 (nested vs. crossed grouping factors, Table 2),
  [doi:10.18637/jss.v067.i01](https://doi.org/10.18637/jss.v067.i01).
- D. Bates, *lme4: Mixed-effects modeling with R* (book draft, 2010), chapter 2,
  "Models with multiple random-effects terms" (Penicillin, Pastes, InstEval) — cited only.
- The `InstEval` model follows lme4's `tests/vcov-etc.R` and `vignette("lmerperf")`, reduced to
  `y ~ service + (1|s) + (1|d) + (1|dept:service)`.

```r
library(lme4)
## ?Penicillin -- fully crossed
xtabs(~ sample + plate, Penicillin)
(fm1 <- lmer(diameter ~ (1|plate) + (1|sample), Penicillin))
L <- getME(fm1, "L")

## ?Pastes -- four equivalent models differing only in specification
(fm1 <- lmer(strength ~ (1|batch) + (1|sample), Pastes))
(fm2 <- lmer(strength ~ (1|batch/cask), Pastes))
(fm3 <- lmer(strength ~ (1|batch) + (1|batch:cask), Pastes))
(fm4 <- lmer(strength ~ (1|batch/sample), Pastes))
head(ranef(fm4)[[1]])   ## redundant labels on the sample:batch interaction
head(ranef(fm1)[[1]])
## This model is different and NOT appropriate for these data
(fm5 <- lmer(strength ~ (1|batch) + (1|cask), Pastes))
with(Pastes, isNested(cask, batch))    ## FALSE
with(Pastes, isNested(sample, batch))  ## TRUE
## book, ch. 2: is the batch variance component needed?
anova(lmer(strength ~ 1 + (1|sample), Pastes, REML = FALSE),
      lmer(strength ~ 1 + (1|sample) + (1|batch), Pastes, REML = FALSE))

## ?InstEval
str(InstEval); xtabs(~ service + dept, InstEval)
system.time(fmI <- lmer(y ~ service + (1|s) + (1|d) + (1|dept:service), InstEval))
```

The R reference values are produced by
[`r/02_crossed_and_nested_random_effects.R`](r/02_crossed_and_nested_random_effects.R)
(R 4.6.2, lme4 2.1.0) and stored in `reference/02_crossed_and_nested_random_effects.json`.

In [1]:
import json
import time

import numpy as np
import pandas as pd
from numpy.testing import assert_allclose

import lme4
from lme4 import lmer, load_dataset, isNested

ref = json.load(open("reference/02_crossed_and_nested_random_effects.json"))
print("R", ref["R_version"], "/ lme4", ref["lme4_version"], "  vs  lme4 for Python", lme4.__version__)


def check_fit(m, r, rtol=1e-6, rtol_crit=1e-8):
    """compare a fitted model with the summary written by fitinfo() in the R script"""
    if m.isREML():
        assert_allclose(m.REMLcrit(), r["REMLcrit"], rtol=rtol_crit)
    else:
        assert_allclose(m.deviance(), r["deviance"], rtol=rtol_crit)
    assert_allclose(float(m.logLik()), r["logLik"], rtol=rtol_crit)
    assert list(m.getME("theta").index) == r["theta_names"]
    assert_allclose(m.theta, r["theta"], rtol=rtol, atol=1e-8)
    assert_allclose(m.sigma(), r["sigma"], rtol=rtol)
    assert_allclose(m.fixef().values, list(r["fixef"].values()), rtol=rtol)
    assert_allclose(np.sqrt(np.diag(m.vcov())), r["se"], rtol=rtol)
    vc = m.VarCorr().as_data_frame()
    assert list(vc["grp"]) == r["vc_grp"]
    assert_allclose(vc["sdcor"], r["vc_sdcor"], rtol=rtol)
    assert dict(m.ngrps()) == r["ngrps"]
    re = m.ranef()
    assert list(re) == list(r["ranef"])
    for g, rr in r["ranef"].items():
        assert list(re[g].index) == rr["levels"], g
        assert_allclose(re[g].iloc[:, 0], rr["values"], rtol=rtol,
                        atol=rtol * np.abs(rr["values"]).max())
    return "ok"

R 4.6.2 / lme4 2.1.0   vs  lme4 for Python 2.1.0a2


## 1. Fully crossed random effects: `Penicillin`

Six samples of penicillin were each assayed on each of 24 plates; the response is the diameter
(mm) of the zone where bacterial growth is inhibited. Every sample appears on every plate
exactly once, so `plate` and `sample` are **fully crossed**.

In [2]:
Penicillin = load_dataset("Penicillin")
print(Penicillin.dtypes)
xt = pd.crosstab(Penicillin["sample"], Penicillin["plate"])
xt

diameter     float64
plate       category
sample      category
dtype: object


plate,a,b,c,d,e,f,g,h,i,j,...,o,p,q,r,s,t,u,v,w,x
sample,,,,,,,,,,,,,,,,,,,,,
A,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
B,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
C,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
D,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
E,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
F,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [3]:
print("every sample on every plate exactly once:", bool((xt.values == 1).all()))
print("plate nested in sample?", isNested(Penicillin["plate"], Penicillin["sample"]))

every sample on every plate exactly once: True
plate nested in sample? False


A model with a random intercept for each plate and one for each sample: the two random-effects
terms are simply added.

In [4]:
fm1 = lmer("diameter ~ (1|plate) + (1|sample)", Penicillin)
print(fm1.summary())

Linear mixed model fit by REML ['lmerMod']
Formula: diameter ~ (1 | plate) + (1 | sample)
   Data: Penicillin

REML criterion at convergence: 330.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.07923 -0.67140  0.06292  0.58377  2.97959 

Random effects:
 Groups   Name        Variance Std.Dev.
 plate    (Intercept) 0.7169   0.8467  
 sample   (Intercept) 3.7311   1.9316  
 Residual             0.3024   0.5499  
Number of obs: 144, groups:  plate, 24; sample, 6

Fixed effects:
            Estimate Std. Error t value
(Intercept)  22.9722     0.8086   28.41


The variability between samples (σ ≈ 1.93) is more than twice that between plates (σ ≈ 0.85).
The conditional modes (BLUPs) of the sample effects:

In [5]:
re = fm1.ranef()
re["sample"].T.round(4)

,A,B,C,D,E,F
(Intercept),2.1871,-1.0105,1.9379,-0.0969,-0.0138,-3.0037


### Check against R

The REML criterion is very flat in the direction of θ₂ = σ_sample/σ (only 6 samples): R and
Python stop at points whose criteria differ by ~4e-9 (Python's is marginally lower), so θ₂ and
the quantities that depend on it agree to ~1e-5 rather than 1e-6. Evaluating this package's
deviance function at R's θ reproduces R's criterion to 1e-12, and the ML fit agrees to ~1e-11.

In [6]:
assert bool((xt.values == 1).all()) == ref["penicillin_xtab_all_one"]
assert isNested(Penicillin["plate"], Penicillin["sample"]) == ref["pastes_isNested"]["plate_in_sample"]
print("REML:", check_fit(fm1, ref["penicillin"], rtol=5e-5))
devfun = fm1.getME("devfun")
assert_allclose(devfun(np.array(ref["penicillin"]["theta"])), ref["penicillin"]["REMLcrit"], rtol=1e-12)
print("ML:  ", check_fit(fm1.update(REML=False), ref["penicillin_ML"]))

REML: ok


ML:   ok


### Structure of the model: `getME()`

The random-effects model matrix is stored transposed, `Zt` = **Z**ᵀ, as a sparse matrix with one
row per random effect (24 plates followed by 6 samples, delimited by the group pointers `Gp`)
and one column per observation. Each column has exactly one non-zero per term — two here.

In [7]:
Zt = fm1.getME("Zt")
print(type(Zt).__name__, Zt.shape, "non-zeros:", Zt.nnz,
      f"(density {Zt.nnz / np.prod(Zt.shape):.3f})")
print("Gp   =", fm1.getME("Gp"), "  cnms =", fm1.getME("cnms"))
print("l_i  =", dict(fm1.getME("l_i")))
print("column sums of Zt:", np.unique(np.asarray(Zt.sum(axis=0)).ravel()))

csc_matrix (30, 144) non-zeros: 288 (density 0.067)
Gp   = [ 0 24 30]   cnms = [('plate', ['(Intercept)']), ('sample', ['(Intercept)'])]
l_i  = {'plate': np.int64(24), 'sample': np.int64(6)}
column sums of Zt: [2.]


In [8]:
# a text "image" of the first 12 observations (columns) of Zt
rows = [f"plate {l}" for l in re["plate"].index] + [f"sample {l}" for l in re["sample"].index]
pd.DataFrame(Zt[:, :12].toarray().astype(int), index=rows).replace(0, ".")

,0,1,2,3,4,5,6,7,8,9,10,11
plate a,1,1,1,1,1,1,.,.,.,.,.,.
plate b,.,.,.,.,.,.,1,1,1,1,1,1
plate c,.,.,.,.,.,.,.,.,.,.,.,.
plate d,.,.,.,.,.,.,.,.,.,.,.,.
plate e,.,.,.,.,.,.,.,.,.,.,.,.
plate f,.,.,.,.,.,.,.,.,.,.,.,.
plate g,.,.,.,.,.,.,.,.,.,.,.,.
plate h,.,.,.,.,.,.,.,.,.,.,.,.
plate i,.,.,.,.,.,.,.,.,.,.,.,.
plate j,.,.,.,.,.,.,.,.,.,.,.,.


`Lambdat` = **Λ**ᵀ, the transposed relative covariance factor, is diagonal for scalar random
effects: θ₁ = σ_plate/σ repeated 24 times, then θ₂ = σ_sample/σ repeated 6 times. `Lind` maps
each non-zero of Λᵀ to an element of θ (1-based, as in R).

In [9]:
Lt = fm1.getME("Lambdat")
print(Lt.shape, "nnz:", Lt.nnz, " theta =", fm1.getME("theta").round(5).to_dict())
print("diag(Lambdat):", np.unique(Lt.diagonal().round(6)))
print("Lind:", fm1.getME("Lind"))

(30, 30) nnz: 30  theta = {'plate.(Intercept)': 1.53968, 'sample.(Intercept)': 3.51249}
diag(Lambdat): [1.539676 3.512492]
Lind: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2]


`L` is the sparse Cholesky factor of Λᵀ**Z**ᵀ**Z**Λ + **I** (after a fill-reducing permutation
**P**). For crossed factors the off-diagonal block Λ₁ᵀZ₁ᵀZ₂Λ₂ is dense (every plate meets every
sample), so `L` is dense in the rows of the sample effects. R uses CHOLMOD and this package
Eigen's `SimplicialLLT` with AMD ordering, so the permutation (and thus the exact non-zero pattern
of `L`) may differ; the permutation-invariant quantities log det(**LL**ᵀ) and
trace(Λᵀ**Z**ᵀ**Z**Λ) are compared with R.

In [10]:
L = fm1.getME("L")
print(L)
logdetL = 2 * np.log(L.L.diagonal()).sum()
print("log det(L L') =", logdetL)
A = Lt @ Zt
LtZtZL = (A @ A.T).toarray()
print("trace(Lambda'Z'Z Lambda) =", np.trace(LtZtZL))
# non-zero pattern of the last 8 rows of L
print(pd.DataFrame(np.abs(L.L.toarray()[-8:]) > 1e-12).replace({True: "x", False: "."})
      .to_string(header=False, index=False))

CHMfactor (dim 30, nnz 189)
log det(L L') = 96.83673345939243
trace(Lambda'Z'Z Lambda) = 2117.9812582178924
. . . . . . . . . . . . . . . . . . . . . . x . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . x . . . . . .
x x x x x x x x x x x x x x x x x x x x x x x x x . . . . .
x x x x x x x x x x x x x x x x x x x x x x x x x x . . . .
x x x x x x x x x x x x x x x x x x x x x x x x x x x . . .
x x x x x x x x x x x x x x x x x x x x x x x x x x x x . .
x x x x x x x x x x x x x x x x x x x x x x x x x x x x x .
x x x x x x x x x x x x x x x x x x x x x x x x x x x x x x


### Check against R

In [11]:
g = ref["penicillin_getME"]
assert list(Zt.shape) == g["Zt_dim"] and Zt.nnz == g["Zt_nnz"]
assert_allclose(np.asarray(Zt.sum(axis=0)).ravel(), g["Zt_colsums"])
assert list(re["plate"].index) + list(re["sample"].index) == g["Zt_rownames"]
assert list(Lt.shape) == g["Lambdat_dim"] and Lt.nnz == g["Lambdat_nnz"]
assert_allclose(Lt.diagonal(), g["Lambdat_diag"], rtol=5e-5)
assert list(fm1.getME("Lind")) == g["Lind"]
assert list(fm1.getME("Gp")) == g["Gp"]
assert dict(fm1.getME("l_i")) == g["l_i"]
assert_allclose(logdetL, g["logdetL"], rtol=1e-5)   # depends on the (flat) theta_2
assert_allclose(np.trace(LtZtZL), g["LtZtZL_trace"], rtol=5e-5)
print("ok")

ok


## 2. Nested random effects: `Pastes`

The strength of a chemical paste was measured twice on each of 3 casks from each of 10 batches.
The casks are labelled `a`, `b`, `c` *within* each batch: cask `a` of batch `A` has nothing to do
with cask `a` of batch `B`. The factor `cask` is therefore only **implicitly nested** in `batch`,
while `sample` (= `batch:cask`, 30 levels) is the **explicitly nested** version.

In [12]:
Pastes = load_dataset("Pastes")
print(Pastes.head(8))
pd.crosstab(Pastes["batch"], Pastes["cask"])

   strength batch cask sample
0      62.8     A    a    A:a
1      62.6     A    a    A:a
2      60.1     A    b    A:b
3      62.3     A    b    A:b
4      62.7     A    c    A:c
5      63.1     A    c    A:c
6      60.0     B    a    B:a
7      61.4     B    a    B:a


cask,a,b,c
batch,,,
A,2,2,2
B,2,2,2
C,2,2,2
D,2,2,2
E,2,2,2
F,2,2,2
G,2,2,2
H,2,2,2
I,2,2,2


`isNested(f1, f2)` (from `reformulas`, re-exported by lme4; added to this package's public API
for this notebook) checks whether each level of `f1` occurs with exactly one level of `f2`:

In [13]:
nested = {
    "cask_in_batch": isNested(Pastes["cask"], Pastes["batch"]),
    "sample_in_batch": isNested(Pastes["sample"], Pastes["batch"]),
    "batch_in_sample": isNested(Pastes["batch"], Pastes["sample"]),
}
nested

{'cask_in_batch': False, 'sample_in_batch': True, 'batch_in_sample': False}

`cask` is not *syntactically* nested in `batch` — its labels are re-used — so `(1|batch) + (1|cask)`
would treat the 3 casks as crossed with the 10 batches. The first four models below are
equivalent ways of writing the nested model; `fm5` is a different model, inappropriate here.

In [14]:
fm_p = {
    "fm1": lmer("strength ~ (1|batch) + (1|sample)", Pastes),        # explicit nesting
    "fm2": lmer("strength ~ (1|batch/cask)", Pastes),                # nesting operator
    "fm3": lmer("strength ~ (1|batch) + (1|batch:cask)", Pastes),    # interaction
    "fm4": lmer("strength ~ (1|batch/sample)", Pastes),              # redundant, but fine
    "fm5": lmer("strength ~ (1|batch) + (1|cask)", Pastes),          # NOT appropriate
}
print(fm_p["fm2"])

Linear mixed model fit by REML ['lmerMod']
Formula: strength ~ (1 | batch/cask)
   Data: Pastes
REML criterion at convergence: 246.9907
Random effects:
 Groups     Name        Std.Dev.
 cask:batch (Intercept) 2.9041  
 batch      (Intercept) 1.2874  
 Residual               0.8234  
Number of obs: 60, groups:  cask:batch, 30; batch, 10
Fixed Effects:
(Intercept)  
      60.05  


In [15]:
def row(m):
    vc = m.VarCorr().as_data_frame()
    return {"terms": " + ".join(f"{g} ({n})" for g, n in m.ngrps().items()),
            "REML crit": m.REMLcrit(), "logLik": float(m.logLik()),
            "sd term 1": vc["sdcor"].iloc[0], "sd term 2": vc["sdcor"].iloc[1], "sigma": m.sigma()}

pd.DataFrame({k: row(m) for k, m in fm_p.items()}).T

,terms,REML crit,logLik,sd term 1,sd term 2,sigma
fm1,sample (30) + batch (10),246.990746,-123.495373,2.904078,1.287365,0.823408
fm2,cask:batch (30) + batch (10),246.990746,-123.495373,2.904078,1.287364,0.823408
fm3,batch:cask (30) + batch (10),246.990746,-123.495373,2.904078,1.287365,0.823408
fm4,sample:batch (30) + batch (10),246.990746,-123.495373,2.904078,1.287365,0.823408
fm5,batch (10) + cask (3),301.470914,-150.735457,1.834086,0.38557,2.702961


`fm1`–`fm4` reach the same REML criterion; `fm5` is markedly worse because it pools cask `a`
over all batches. `(1|batch/cask)` expands to `(1|batch) + (1|cask:batch)`; `(1|batch/sample)`
gives a term `sample:batch` whose labels repeat the batch (`A:a:A`):

In [16]:
print(fm_p["fm4"].ranef()["sample:batch"].head(6))
print(fm_p["fm1"].ranef()["sample"].head(6))

       (Intercept)
A:a:A     1.774687
A:b:A     0.332651
A:c:A     1.966958
B:a:B     0.883656
B:b:B    -2.481095
B:c:B     0.210706
     (Intercept)
A:a     1.774687
A:b     0.332651
A:c     1.966958
B:a     0.883656
B:b    -2.481095
B:c     0.210706


From the book draft (ch. 2): is the between-batch variance needed once samples are in the model?
A likelihood-ratio test on ML fits:

In [17]:
p1ML = lmer("strength ~ 1 + (1|sample) + (1|batch)", Pastes, REML=False)
p1aML = lmer("strength ~ 1 + (1|sample)", Pastes, REML=False)
aa = p1aML.anova(p1ML)
aa

,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
MODEL1,3,254.401700,260.684733,-124.200850,248.401700,NaN,NaN,NaN
MODEL2,4,255.994466,264.371844,-123.997233,247.994466,0.407234,1.0,0.523377


The p-value is conservative — the null value σ²_batch = 0 lies on the boundary of the parameter
space — but either way there is little evidence of batch-to-batch variability beyond the
sample-to-sample variability.

### Check against R

The four equivalent models reach the same optimum; `fm2` has a different term labelling and
the optimizer takes a slightly different path, so its θ agrees with `fm1` only to ~1e-6 — but
each model agrees with its own R counterpart: criteria to 1e-8, estimates to ~1e-6 (checked with
rtol=1e-5, since the 10-batch variance is weakly determined).

In [18]:
assert nested == {k: ref["pastes_isNested"][k] for k in nested}
for k, m in fm_p.items():
    print(k, check_fit(m, ref["pastes"][k], rtol=1e-5))
for name, r_ in [("fm4", fm_p["fm4"].ranef()["sample:batch"]), ("fm1", fm_p["fm1"].ranef()["sample"])]:
    r = ref[f"pastes_head_ranef_{name}"]
    assert list(r_.index[:6]) == r["levels"]
    assert_allclose(r_.iloc[:6, 0], r["values"], rtol=1e-5)
ra = ref["pastes_anova"]
for col, key in [("npar", "npar"), ("AIC", "AIC"), ("BIC", "BIC"), ("logLik", "logLik"),
                 ("-2*log(L)", "minus2logL")]:
    assert_allclose(aa[col].astype(float), ra[key], rtol=1e-8)
assert_allclose(aa["Chisq"].iloc[1], ra["Chisq"][1], rtol=1e-6)
assert_allclose(aa["Pr(>Chisq)"].iloc[1], ra["p"][1], rtol=1e-6)
print("ok")

fm1 ok
fm2 ok
fm3 ok


fm4 ok
fm5 ok
ok


## 3. A large, partially crossed design: `InstEval`

University lecture evaluations at ETH Zürich: 73 421 ratings `y` (1–5) of 1128 lecturers `d` by
2972 students `s`. Students and lecturers are **partially crossed** (each student rates only a
few lecturers). `dept` is the lecturer's department and `service` whether the lecture was a
service course for another department; `dept:service` has 28 combinations.

In [19]:
InstEval = load_dataset("InstEval")
print(InstEval.shape)
print(InstEval.dtypes)
InstEval.head(16)

(73421, 7)
s          category
d          category
studage    category
lectage    category
service    category
dept       category
y             int64
dtype: object


,s,d,studage,lectage,service,dept,y
0,1,1002,2,2,0,2,5
1,1,1050,2,1,1,6,2
2,1,1582,2,2,0,2,5
3,1,2050,2,2,1,3,3
4,2,115,2,1,0,5,2
5,2,756,2,1,0,5,4
6,3,7,2,1,1,11,4
7,3,13,2,1,0,10,5
8,3,36,2,1,0,10,5
9,3,140,2,1,0,10,4


In [20]:
pd.crosstab(InstEval["service"], InstEval["dept"])

dept,15,5,10,12,6,7,4,8,9,14,1,3,11,2
service,,,,,,,,,,,,,,
0,2466,3576,4343,6209,3772,1601,4518,578,4224,2606,1260,3550,1711,1224
1,826,214,365,3319,4325,919,2207,3848,2400,1328,1372,1199,6863,2598


In [21]:
print("lecturer nested in department:", isNested(InstEval["d"], InstEval["dept"]))
print("student nested in lecturer:   ", isNested(InstEval["s"], InstEval["d"]))

lecturer nested in department: True
student nested in lecturer:    False


In [22]:
t0 = time.perf_counter()
fmI = lmer("y ~ service + (1|s) + (1|d) + (1|dept:service)", InstEval)
elapsed = time.perf_counter() - t0
print(fmI.summary())
print(f"\nfit time: {elapsed:.1f} s (Python, this run);  R reference run: {ref['insteval_time']:.1f} s")

Linear mixed model fit by REML ['lmerMod']
Formula: y ~ service + (1 | s) + (1 | d) + (1 | dept:service)
   Data: InstEval

REML criterion at convergence: 237661.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.99701 -0.74750  0.04017  0.77230  3.11722 

Random effects:
 Groups       Name        Variance Std.Dev.
 s            (Intercept) 0.10543  0.3247  
 d            (Intercept) 0.26257  0.5124  
 dept:service (Intercept) 0.01202  0.1097  
 Residual                 1.38496  1.1768  
Number of obs: 73421, groups:  s, 2972; d, 1128; dept:service, 28

Fixed effects:
            Estimate Std. Error t value
(Intercept)  3.28067    0.03526  93.033
service1    -0.05350    0.04454  -1.201

Correlation of Fixed Effects:
         (Intr)
service1 -0.601

fit time: 37.7 s (Python, this run);  R reference run: 367.2 s


Timings depend on the machine and load and are shown only for orientation (R uses CHOLMOD's
supernodal Cholesky; this package uses Eigen's simplicial Cholesky). **Z**ᵀ has
4128 rows (2972 + 1128 + 28 random effects) and three non-zeros per column.

In [23]:
ZtI = fmI.getME("Zt")
LI = fmI.getME("L")
print("Zt:", ZtI.shape, "nnz", ZtI.nnz, f"density {ZtI.nnz / np.prod(ZtI.shape):.2e}")
print("Gp:", fmI.getME("Gp"))
print("distinct diag(Lambdat):", np.unique(fmI.getME("Lambdat").diagonal().round(6)))
print(LI, "  log det(L L') =", 2 * np.log(LI.L.diagonal()).sum())

Zt: (4128, 73421) nnz 220263 density 7.27e-04
Gp: [   0 2972 4100 4128]
distinct diag(Lambdat): [0.093178 0.275904 0.435414]
CHMfactor (dim 4128, nnz 577582)   log det(L L') = 5382.980517036745


### Check against R

The REML criterion agrees with R to ~1e-13 (Python's optimum is lower by 2e-8). The surface is
flat in θ for the 28-level `dept:service` term: θ₃ differs by 2e-5 relative, which moves the
standard errors and the `dept:service` modes by about 1e-5. θ, β, σ, standard errors and all
4128 conditional modes are therefore compared with `rtol=5e-5`.

In [24]:
assert list(InstEval.shape) == ref["insteval_dim"]
assert_allclose(pd.crosstab(InstEval["service"], InstEval["dept"]).values.ravel(order="F"),
                ref["insteval_xtab"])
assert isNested(InstEval["d"], InstEval["dept"]) == ref["insteval_isNested"]["d_in_dept"]
assert isNested(InstEval["s"], InstEval["d"]) == ref["insteval_isNested"]["s_in_d"]
print(check_fit(fmI, ref["insteval"], rtol=5e-5))
gI = ref["insteval_getME"]
assert list(ZtI.shape) == gI["Zt_dim"] and ZtI.nnz == gI["Zt_nnz"]
assert list(fmI.getME("Gp")) == gI["Gp"]
assert_allclose(np.unique(fmI.getME("Lambdat").diagonal()), np.sort(gI["Lambdat_diag_unique"]),
                rtol=5e-5)
assert_allclose(2 * np.log(LI.L.diagonal()).sum(), gI["logdetL"], rtol=1e-6)
print("ok")

ok
ok


## Summary

| design | grouping factors | formula |
|---|---|---|
| fully crossed | `plate` × `sample` | `(1|plate) + (1|sample)` |
| nested, unique labels | `sample` within `batch` | `(1|batch) + (1|sample)` |
| nested, re-used labels | `cask` within `batch` | `(1|batch/cask)` = `(1|batch) + (1|batch:cask)` |
| partially crossed | `s`, `d`, `dept:service` | `(1|s) + (1|d) + (1|dept:service)` |

lme4 does not need to be told whether factors are nested or crossed — the sparse Cholesky
factorization handles both. What matters is that the *labels* identify the groups: with re-used
labels (`cask`) write `batch/cask` or `batch:cask`; `isNested()` tells which case one is in.